In [1]:
import pandas as pd
import numpy as np
import os

print("Library dimuat")

Library dimuat


In [2]:
# 1. Load Data Gabungan Annotator
combined_path = '../outputs/evaluation/combined_annotations.csv'
df_combined = pd.read_csv(combined_path)

# Identifikasi kolom skor
score_cols = [col for col in df_combined.columns if col.startswith('score_')]

# Paksa konversi ke numeric (antisipasi jika terbaca sebagai string/object)
df_combined[score_cols] = df_combined[score_cols].apply(pd.to_numeric, errors='coerce')

print(f"Data gabungan annotator dimuat: {len(df_combined)} baris")
print(f"Kolom skor yang terdeteksi: {score_cols}")

Data gabungan annotator dimuat: 450 baris
Kolom skor yang terdeteksi: ['score_annotator 1', 'score_annotator 2', 'score_annotator 3', 'score_annotator 4', 'score_annotator 5']


In [3]:
# 2. Hitung Standard Deviation dan Mean per Tweet
df_combined['sd_annotator'] = df_combined[score_cols].std(axis=1, ddof=1)
df_combined['ground_truth_mean'] = df_combined[score_cols].mean(axis=1)

print("Perhitungan Mean dan SD selesai.")

Perhitungan Mean dan SD selesai.


In [4]:
# 3. Filter Data Berdasarkan Ambang Batas VADER (SD < 2.5)
df_reliable = df_combined[df_combined['sd_annotator'] < 2.5].copy()

print(f"Data setelah filter (SD < 2.5): {len(df_reliable)} dari {len(df_combined)} tweet")

Data setelah filter (SD < 2.5): 450 dari 450 tweet


In [25]:
# 4. Normalisasi Skor Mean ke Skala VADER (-1 sampai +1)
# Skala asli [-4, 4] dibagi 4 agar sejajar dengan compound score VADER
df_reliable['ground_truth_scaled'] = df_reliable['ground_truth_mean'] / 4.0

print("Normalisasi skor ke skala VADER selesai.")

Normalisasi skor ke skala VADER selesai.


In [ ]:
# 5. Tambahkan Kolom Kategori Final (threshold standar VADER)
raw_data_path = '../../datapreparation/datapreprocessingcopy/data_preprocessing_final.csv'
df_raw = pd.read_csv(raw_data_path)

# Pastikan kolom 'no' bertipe sama agar merge tidak gagal
df_raw['no'] = df_raw['no'].astype(str)
df_reliable['no'] = df_reliable['no'].astype(str)

# Merge hanya mengambil tweet yang lolos filter SD
df_final = pd.merge(
    df_raw, 
    df_reliable[['no', 'ground_truth_mean', 'ground_truth_scaled', 'sd_annotator']], 
    on='no', 
    how='inner'
)

In [27]:
# 6. Tambahkan Kolom Kategori Final (berdasarkan threshold standar VADER)
def get_final_category(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'
    

df_final['ground_truth_category'] = df_final['ground_truth_scaled'].apply(get_final_category)

print(f"Data final dengan teks lengkap: {len(df_final)} tweets")
print("Kategori final berdasarkan konsensus skor berhasil ditambahkan.")
print("-" * 30)
print("Distribusi Kategori Ground Truth:")
print(df_final['ground_truth_category'].value_counts())

Data final dengan teks lengkap: 450 tweets
Kategori final berdasarkan konsensus skor berhasil ditambahkan.
------------------------------
Distribusi Kategori Ground Truth:
ground_truth_category
negative    170
positive    157
neutral     123
Name: count, dtype: int64


In [28]:
os.makedirs('../outputs/evaluation', exist_ok=True)
output_path = '../outputs/evaluation/ground_truth_final.csv'

# utf-8-sig dipakai agar file CSV bisa dibuka rapi di Excel tanpa error karakter
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"File berhasil disimpan di: {output_path}")

File berhasil disimpan di: ../outputs/evaluation/ground_truth_final.csv
